### 1) Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

/Users/rishabh013/prod_new/genai/langchain-langgraph/langchainupdated/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots and their ability to mimic human speech. Parrots are known for their vocal mimicry, which is a key part of their communication. I think it might be related to their social behavior. In the wild, parrots live in flocks, so maybe they use vocalizations to communicate with each other.\n\nI remember reading that parrots have a syrinx, which is their vocal organ, different from the human larynx. This structure allows them to produce a wide range of sounds. So their anatomy plays a role. Also, some parrots are more vocal than others. Macaws, for example, are known to be good talkers, while others like cockatiels might not be as prominent.\n\nAnother angle is the social aspect. Parrots form strong bonds with their human caregivers, and talking might be a way they interact or gain attention. They might learn to associate certain words with positive outcomes, like gettin

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'smb3aavp7', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.149177218, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.007334567, 'prompt_tokens_details': None, 'queue_time': 0.159827002, 'total_time': 0.156511785}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--1112ae3f-e786-4bca-b

### 2) Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is currently sunny.


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there are no other parameters needed. The required field is just location, so that\'s all I need. Alright, the tool call should be correct.\n', 'tool_calls': [{'id': 'n8ehy9ze8', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 153, 'total_tokens': 258, 'completion_time': 0.150715187, 'completion_tokens_details': {'reasoning_tokens': 81}, 'prompt_time': 0.007019378, 'prompt_tokens_details': None, 'queue_time': 0.056144882, 'total_time': 0.157734565}, 'model_name': 'qwen/qwen3-32b', 'system_fingerpri